# NAFNet-Alpha — Demo Notebook

Controllable smartphone image denoising. This notebook clones the repository, downloads the final checkpoint, and runs it on your own images at different alpha values.

Alpha behavior: `0.00` maximum denoising, `0.20` natural balanced output (recommended), `0.50` moderate grain retention, `1.00` strongest grain retention.

Run the cells in order. Designed for Google Colab (GPU runtime recommended: Runtime -> Change runtime type -> T4 GPU).

## 1. Get the code and checkpoint

In [ ]:
!git clone https://github.com/borhanibarbod/NAFNet-Alpha.git /content/NAFNet-Alpha
!wget -q -O /content/NAFNet-Alpha/checkpoints/EXPB_FINAL_a0_42.60.pth \
    https://github.com/borhanibarbod/NAFNet-Alpha/releases/download/v1.0/EXPB_FINAL_a0_42.60.pth
print("Repository cloned and checkpoint downloaded.")

In [ ]:
NAF_DIR = "/content/NAFNet-Alpha"
CKPT = NAF_DIR + "/checkpoints/EXPB_FINAL_a0_42.60.pth"

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 80)
print("NAFNet-Alpha Final Model — Demo Notebook")
print("=" * 80)
print(f"Device: {DEVICE}\n")

sys.path.insert(0, NAF_DIR + "/src")
if "model" in sys.modules:
    del sys.modules["model"]

from model import NAFNetAlpha, run_model, tensor_to_pil, pil_to_tensor, collect_images

## 2. Load the model

In [ ]:
model = NAFNetAlpha(width=32).to(DEVICE)
ck = torch.load(CKPT, map_location=DEVICE)
state = ck.get("model_state", ck)
model.load_state_dict(state, strict=False)
model.eval()

print(f"Checkpoint loaded: {Path(CKPT).name}")
print(f"  alpha=0: 42.60 dB | alpha=0.5: 38.85 dB | alpha=1: 34.64 dB (internal SIDD validation)\n")

ALPHA_SWEEP = [0.0, 0.15, 0.30, 0.45, 0.60, 0.75, 1.00]
OUTPUT_MODES = {
    "clean":   {"alpha": 0.0, "desc": "Maximum denoising"},
    "natural": {"alpha": 0.2, "desc": "Balanced (recommended)"},
    "grain":   {"alpha": 0.5, "desc": "Moderate grain"},
}

## 3. Helper functions

In [ ]:
def center_crop(img, ratio=0.4):
    w, h = img.size
    cw, ch = int(w * ratio), int(h * ratio)
    left, top = (w - cw) // 2, (h - ch) // 2
    return img.crop((left, top, left + cw, top + ch))


def save_grid(images, titles, path, ncols=4):
    n = len(images)
    nrows = (n + ncols - 1) // ncols
    fig = plt.figure(figsize=(ncols * 3, nrows * 3))
    for i, (img, title) in enumerate(zip(images, titles), 1):
        ax = fig.add_subplot(nrows, ncols, i)
        ax.imshow(img)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"  Saved: {Path(path).name}")

## 4. Upload test images

Upload one or more noisy photos (or a ZIP of photos).

In [ ]:
from google.colab import files

print("Upload your test images (or ZIP file)")
uploaded = files.upload()

test_dir = "/content/test_images"
os.makedirs(test_dir, exist_ok=True)

for name in uploaded.keys():
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name, "r") as z:
            z.extractall(test_dir)
    else:
        shutil.copy(name, test_dir)

img_paths = collect_images(test_dir)
assert img_paths, "No images found!"
print(f"\nFound {len(img_paths)} images\n")

## 5. Run: alpha sweep, crop detail, and output modes

For every image this saves the original, one output per alpha value, a full sweep grid, a center-crop detail grid, and the named output modes.

Results are saved inside the Colab session at `/content/NAFNet-Alpha/results/final_validation`. This storage is temporary — download the `results` folder before the session ends if you want to keep the outputs (the last cell does this automatically as a ZIP).

In [ ]:
output_root = os.path.join(NAF_DIR, "results", "final_validation")
os.makedirs(output_root, exist_ok=True)

for img_path in img_paths:
    img_name = Path(img_path).stem
    img_out_dir = os.path.join(output_root, img_name)
    os.makedirs(img_out_dir, exist_ok=True)

    print(f"Testing: {img_name}")

    img = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    w, h = img.size
    if max(w, h) > 2048:
        scale = 2048 / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

    img.save(os.path.join(img_out_dir, f"{img_name}_00_original.png"))

    # Alpha sweep
    print("  Alpha sweep...")
    sweep_imgs, sweep_titles = [img], ["Original"]
    for alpha_val in ALPHA_SWEEP:
        result = run_model(model, img, alpha=alpha_val, device=DEVICE)
        sweep_imgs.append(result)
        sweep_titles.append(f"alpha={alpha_val:.2f}")
        result.save(os.path.join(img_out_dir, f"{img_name}_alpha_{alpha_val:.2f}.png"))

    save_grid(sweep_imgs, sweep_titles,
              os.path.join(img_out_dir, f"{img_name}_01_alpha_sweep.png"), ncols=4)

    # Crop detail
    print("  Crop detail...")
    crop_sweep, crop_titles = [center_crop(img)], ["Original"]
    for alpha_val in ALPHA_SWEEP:
        c = center_crop(sweep_imgs[ALPHA_SWEEP.index(alpha_val) + 1])
        crop_sweep.append(c)
        crop_titles.append(f"alpha={alpha_val:.2f}")

    save_grid(crop_sweep, crop_titles,
              os.path.join(img_out_dir, f"{img_name}_01_alpha_sweep_crop.png"), ncols=4)

    # Output modes
    print("  Output modes...")
    mode_imgs, mode_titles = [img], ["Original"]
    for mode_name, params in OUTPUT_MODES.items():
        result = run_model(model, img, alpha=params["alpha"], device=DEVICE)
        mode_imgs.append(result)
        mode_titles.append(mode_name)
        result.save(os.path.join(img_out_dir, f"{img_name}_{mode_name}.png"))

    save_grid(mode_imgs, mode_titles,
              os.path.join(img_out_dir, f"{img_name}_02_modes.png"), ncols=3)

    print(f"  Done: {img_out_dir}\n")

print("=" * 80)
print("All tests complete. Results saved to:")
print(output_root)
print("=" * 80)

## 6. Download results

Zips the results folder and downloads it to your computer, since Colab storage is temporary.

In [ ]:
import shutil as _shutil
zip_path = "/content/NAFNet-Alpha-results"
_shutil.make_archive(zip_path, "zip", output_root)
files.download(zip_path + ".zip")